# Photoisomerization: Excited-State Potential Energy Surfaces

**Pipeline**: Molecular geometry → Hamiltonian at each torsion angle → Ground & excited state energies (uniqx) → PES plot

Cis-trans photoisomerization is one of the most fundamental photochemical reactions. A photon promotes the molecule from the ground state (S0) to an excited state (S1), the molecule twists along a torsion coordinate, passes through a conical intersection (CI) where S0 and S1 nearly touch, and decays back to S0 — landing in either the cis or trans configuration.

This notebook computes the potential energy surface (PES) for torsional isomerization of small conjugated molecules, visualising:
- **S0 and S1 energy curves** vs torsion angle
- **Franck-Condon region** (vertical excitation at equilibrium geometry)
- **Conical intersection** (where S0 and S1 approach each other)
- **Dissociation of excitation energy** across the reaction coordinate

Supported molecules: ethylene (H₂C=CH₂), formaldimine (H₂C=NH), butadiene (C₄H₆), acrolein (CH₂=CHCHO)

## 1. Setup & Configuration

In [ ]:
import os
from uniqx import connect

# Default to a local service for development.
# For prod, export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "localhost:50050")
client = connect(endpoint)
print("Connected to", endpoint)
import math
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx import ops, tracing, fmt_mat, parse_result
from uniqx.ops import SX, SZ, I2
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")

# ----- User-configurable parameters -----
MOLECULE = "ethylene"  # ethylene | formaldimine | butadiene | acrolein
BASIS = "sto-3g"  # sto-3g | 6-31g | cc-pvdz
N_EXCITED = 1  # Number of excited states (the 2-state torsional model has S0 and S1)
ANGLE_STEP = 5  # Torsion angle step size in degrees (5, 10, or 15)

print(f"Molecule:       {MOLECULE}")
print(f"Basis set:      {BASIS}")
print(f"Excited states: {N_EXCITED}")
print(f"Angle step:     {ANGLE_STEP} deg")


## 2. Molecule Definitions

Each molecule is defined by its equilibrium geometry, number of qubits for the
active space, and the torsional Hamiltonian parameters.

The torsional PES is modelled using a **two-state Hamiltonian** whose matrix
elements depend on the dihedral angle $\theta$:

$$H(\theta) = \begin{pmatrix} E_{S0}(\theta) & V(\theta) \\ V(\theta) & E_{S1}^{\text{dia}}(\theta) \end{pmatrix}$$

where the **diabatic** energies follow smooth torsional potentials and $V(\theta)$
is the non-adiabatic coupling that produces the avoided crossing.
Diagonalising $H(\theta)$ at each angle gives the **adiabatic** S0 and S1 surfaces.

In [ ]:
# Molecule database: geometry + Hamiltonian parameters per basis set
#
# Parameters:
#   n_qubits:  active-space qubit count
#   barrier:   S0 torsional barrier at 90 deg (eV)
#   excitation: S0->S1 vertical excitation at 0 deg (eV)
#   s1_min:    S1 energy at ~90 deg (eV) -- must be > barrier for avoided crossing
#   coupling:  non-adiabatic coupling strength V at the CI (eV)
#   s2_offset: S2 vertical excitation above S1 (eV), if applicable
#   trans_offset: energy offset at 180 deg vs 0 deg (eV, negative = trans more stable)

MOLECULE_DB = {
    "ethylene": {
        "label": "Ethylene (H\u2082C=CH\u2082)",
        "formula": "C2H4",
        "atoms": [
            ("C", [0.0, 0.0, 0.665]),
            ("C", [0.0, 0.0, -0.665]),
            ("H", [0.0, 0.923, 1.232]),
            ("H", [0.0, -0.923, 1.232]),
            ("H", [0.0, 0.923, -1.232]),
            ("H", [0.0, -0.923, -1.232]),
        ],
        "n_qubits": 4,
        "description": "Canonical \u03c0\u2192\u03c0* photoisomerization",
        "ci_angle": 90,
        "params": {
            "sto-3g": {
                "barrier": 4.20,
                "excitation": 8.00,
                "s1_min": 4.50,
                "coupling": 0.15,
                "s2_offset": 1.80,
                "trans_offset": -0.05,
            },
            "6-31g": {
                "barrier": 4.10,
                "excitation": 7.80,
                "s1_min": 4.40,
                "coupling": 0.15,
                "s2_offset": 1.75,
                "trans_offset": -0.05,
            },
            "cc-pvdz": {
                "barrier": 4.00,
                "excitation": 7.65,
                "s1_min": 4.28,
                "coupling": 0.14,
                "s2_offset": 1.70,
                "trans_offset": -0.05,
            },
        },
    },
    "formaldimine": {
        "label": "Formaldimine (H\u2082C=NH)",
        "formula": "H2CNH",
        "atoms": [
            ("C", [0.0, 0.0, 0.0]),
            ("N", [0.0, 0.0, 1.267]),
            ("H", [0.0, 0.934, -0.587]),
            ("H", [0.0, -0.934, -0.587]),
            ("H", [0.0, 0.870, 1.740]),
        ],
        "n_qubits": 4,
        "description": "Minimal retinal Schiff base model \u2014 chromophore of biological vision",
        "ci_angle": 88,
        "params": {
            "sto-3g": {
                "barrier": 3.60,
                "excitation": 5.80,
                "s1_min": 3.90,
                "coupling": 0.15,
                "s2_offset": 2.40,
                "trans_offset": -0.12,
            },
            "6-31g": {
                "barrier": 3.50,
                "excitation": 5.60,
                "s1_min": 3.80,
                "coupling": 0.15,
                "s2_offset": 2.40,
                "trans_offset": -0.12,
            },
            "cc-pvdz": {
                "barrier": 3.40,
                "excitation": 5.45,
                "s1_min": 3.70,
                "coupling": 0.14,
                "s2_offset": 2.40,
                "trans_offset": -0.12,
            },
        },
    },
    "butadiene": {
        "label": "Butadiene (C\u2084H\u2086)",
        "formula": "C4H6",
        "atoms": [
            ("C", [-1.83, 0.0, -0.12]),
            ("C", [-0.61, 0.0, 0.12]),
            ("C", [0.61, 0.0, -0.12]),
            ("C", [1.83, 0.0, 0.12]),
            ("H", [-2.72, 0.0, 0.50]),
            ("H", [-1.95, 0.0, -1.19]),
            ("H", [-0.49, 0.0, 1.19]),
            ("H", [0.49, 0.0, -1.19]),
            ("H", [1.95, 0.0, 1.19]),
            ("H", [2.72, 0.0, -0.50]),
        ],
        "n_qubits": 6,
        "description": "Extended \u03c0-conjugation lowers the gap and shifts the CI geometry",
        "ci_angle": 85,
        "params": {
            "sto-3g": {
                "barrier": 4.60,
                "excitation": 6.40,
                "s1_min": 4.80,
                "coupling": 0.10,
                "s2_offset": 1.40,
                "trans_offset": -0.08,
            },
            "6-31g": {
                "barrier": 4.50,
                "excitation": 6.20,
                "s1_min": 4.70,
                "coupling": 0.10,
                "s2_offset": 1.40,
                "trans_offset": -0.08,
            },
            "cc-pvdz": {
                "barrier": 4.40,
                "excitation": 6.05,
                "s1_min": 4.58,
                "coupling": 0.09,
                "s2_offset": 1.40,
                "trans_offset": -0.08,
            },
        },
    },
    "acrolein": {
        "label": "Acrolein (CH\u2082=CHCHO)",
        "formula": "C3H4O",
        "atoms": [
            ("C", [0.0, 0.0, 0.0]),
            ("C", [0.0, 0.0, 1.34]),
            ("C", [0.0, 0.0, 2.64]),
            ("O", [0.0, 0.0, 3.83]),
            ("H", [0.0, 0.93, -0.54]),
            ("H", [0.0, -0.93, -0.54]),
            ("H", [0.0, 0.93, 1.88]),
            ("H", [0.0, -0.93, 2.10]),
        ],
        "n_qubits": 6,
        "description": "Competing n\u2192\u03c0* and \u03c0\u2192\u03c0* pathways with tight avoided crossing",
        "ci_angle": 92,
        "params": {
            "sto-3g": {
                "barrier": 3.00,
                "excitation": 3.80,
                "s1_min": 3.18,
                "coupling": 0.09,
                "s2_offset": 2.80,
                "trans_offset": -0.10,
            },
            "6-31g": {
                "barrier": 2.90,
                "excitation": 3.65,
                "s1_min": 3.08,
                "coupling": 0.09,
                "s2_offset": 2.75,
                "trans_offset": -0.10,
            },
            "cc-pvdz": {
                "barrier": 2.82,
                "excitation": 3.52,
                "s1_min": 3.00,
                "coupling": 0.09,
                "s2_offset": 2.70,
                "trans_offset": -0.10,
            },
        },
    },
}

mol = MOLECULE_DB[MOLECULE]
p = mol["params"][BASIS]
n_qubits = mol["n_qubits"]
dim = 2**n_qubits

print(f"Molecule: {mol['label']}")
print(f"Description: {mol['description']}")
print(f"Active space: {n_qubits} qubits (dim={dim})")
print(f"CI angle: ~{mol['ci_angle']} deg")
print(f"\nParameters ({BASIS}):")
for k, v in p.items():
    print(f"  {k}: {v}")
print(f"\nAtoms ({len(mol['atoms'])})")
for sym, coords in mol["atoms"]:
    print(f"  {sym} ({coords[0]:.3f}, {coords[1]:.3f}, {coords[2]:.3f}) A")

## 3. Torsional Hamiltonian Construction

At each torsion angle $\theta$, we build a model Hamiltonian for the active space.
The diagonal elements encode the **diabatic** state energies (smooth functions of $\theta$),
and the off-diagonal element is the **non-adiabatic coupling** $V(\theta)$ that produces
the avoided crossing.

| Primitive | Classical | Quantum |
|:----------|:----------|:--------|
| `eigs(H, k)` | LAPACK / Lanczos (CPU/GPU) | QPE / VQD (QPU) |
| `expv(H, psi, t)` | Dense matrix exponential (CPU/GPU) | Trotterised circuit (QPU) |
| `expect(O, psi)` | $\psi^\dagger O \psi$ (CPU/GPU) | Pauli measurement (QPU) |

In [ ]:
# --- Pure-Python matrix helpers (used in traced IR, no numpy) ---


def kron(A, B):
    """Kronecker product of two matrices."""
    rA, rB = len(A), len(B)
    n = rA * rB
    C = [[0.0] * n for _ in range(n)]
    for i in range(rA):
        for j in range(rA):
            for k in range(rB):
                for l in range(rB):
                    C[i * rB + k][j * rB + l] = A[i][j] * B[k][l]
    return C


def eye(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def matadd(A, B):
    return [[A[i][j] + B[i][j] for j in range(len(A))] for i in range(len(A))]


def matscale(s, A):
    return [[s * x for x in row] for row in A]


def embed_local(op, qubit, n_qubits):
    result = eye(1)
    for k in range(n_qubits):
        result = kron(result, op if k == qubit else I2)
    return result


def two_body_local(opA, opB, qa, qb, n_qubits):
    parts = []
    for k in range(n_qubits):
        if k == qa:
            parts.append(opA)
        elif k == qb:
            parts.append(opB)
        else:
            parts.append(I2)
    result = eye(1)
    for part in parts:
        result = kron(result, part)
    return result


def build_torsion_hamiltonian(theta_deg, p, n_qubits):
    """Build the active-space Hamiltonian at torsion angle theta.

    Diabatic energies:
      E_s0_dia(theta) = barrier * sin^2(theta) + trans_offset * (theta/180)
      E_s1_dia(theta) = excitation - (excitation - s1_min) * sin^2(theta)

    Non-adiabatic coupling V(theta) peaks at the CI angle.
    The full Hamiltonian is embedded in the qubit active space.
    """
    theta_rad = math.radians(theta_deg)
    sin2 = math.sin(theta_rad) ** 2
    tilt = p["trans_offset"] * (theta_deg / 180.0)

    # Diabatic state energies
    e_s0 = p["barrier"] * sin2 + tilt
    e_s1 = p["excitation"] - (p["excitation"] - p["s1_min"]) * sin2 + tilt * 0.3

    # Coupling: Gaussian centred on the CI region
    coupling = p["coupling"] * math.exp(-0.5 * ((theta_deg - 90.0) / 30.0) ** 2)

    # Build qubit Hamiltonian:
    # H = e_s0 * |0><0| + e_s1 * |1><1| + coupling * (|0><1| + |1><0|)
    # Embedded: e_s0 * (I+Z0)/2 + e_s1 * (I-Z0)/2 + coupling * X0
    dim = 2**n_qubits
    avg = (e_s0 + e_s1) / 2.0
    diff = (e_s0 - e_s1) / 2.0

    H = matscale(avg, eye(dim))
    H = matadd(H, matscale(diff, embed_local(SZ, 0, n_qubits)))
    H = matadd(H, matscale(coupling, embed_local(SX, 0, n_qubits)))

    # Add inter-qubit interactions for richer active-space structure
    for q in range(1, n_qubits):
        scale = 0.05 * sin2 / q  # weak coupling, angle-dependent
        H = matadd(H, matscale(scale, two_body_local(SZ, SZ, 0, q, n_qubits)))

    return H, dim, e_s0, e_s1, coupling


# Test at equilibrium (0 deg)
H_test, _, e0, e1, V = build_torsion_hamiltonian(0.0, p, n_qubits)
print("At theta=0 deg (Franck-Condon):")
print(f"  E_S0 (diabatic) = {e0:.3f} eV")
print(f"  E_S1 (diabatic) = {e1:.3f} eV")
print(f"  Coupling V      = {V:.4f} eV")
print(f"  Vertical gap    = {e1 - e0:.3f} eV  ({1239.8 / (e1 - e0):.0f} nm)")

# Test at CI angle
H_ci, _, e0_ci, e1_ci, V_ci = build_torsion_hamiltonian(
    float(mol["ci_angle"]), p, n_qubits
)
print(f"\nAt theta={mol['ci_angle']} deg (near CI):")
print(f"  E_S0 (diabatic) = {e0_ci:.3f} eV")
print(f"  E_S1 (diabatic) = {e1_ci:.3f} eV")
print(f"  Coupling V      = {V_ci:.4f} eV")
print(f"  Diabatic gap    = {abs(e1_ci - e0_ci):.3f} eV")


def distinct_energy_levels(evals, n_levels):
    """Cluster eigenvalues into distinct energy bands.

    The multi-qubit Hamiltonian embeds 2 electronic states on qubit 0 while
    qubits 1..N are weakly coupled spectators, creating large degeneracies.
    We split the sorted spectrum at the largest gaps to identify bands, then
    return the centroid of each band.
    """
    import numpy as _np
    sorted_evals = _np.sort(evals)
    n = len(sorted_evals)
    if n_levels == 1:
        return [float(_np.mean(sorted_evals))]
    gaps = [(sorted_evals[i + 1] - sorted_evals[i], i) for i in range(n - 1)]
    gaps.sort(reverse=True)
    split_points = sorted([g[1] for g in gaps[:n_levels - 1]])
    levels = []
    start = 0
    for sp in split_points:
        levels.append(float(_np.mean(sorted_evals[start:sp + 1])))
        start = sp + 1
    levels.append(float(_np.mean(sorted_evals[start:])))
    return levels[:n_levels]

## 4. Trace the Eigensolve Module

For each geometry, we diagonalise the Hamiltonian using `ops.eigs` to get the
lowest adiabatic eigenvalues. uniqx routes this to LAPACK (CPU), cuSOLVER (GPU),
or QPE (QPU) depending on the system size and available hardware.

In [ ]:
n_states = min(N_EXCITED + 1, dim)  # +1 for ground state
k_request = dim  # request all eigenvalues to cluster into distinct energy bands


@tracing.to_module(name="pes_eigsolve")
def pes_eigsolve(H_mat):
    """Compute the lowest n_states eigenvalues of the torsional Hamiltonian."""
    eigenvalues, eigenvectors = ops.eigs(
        H_mat, k=k_request, hermitian=True, which="smallest"
    )
    return eigenvalues, eigenvectors


# Trace with test Hamiltonian
mod_test = pes_eigsolve(H_test)
ir_text = mod_test.to_text()
n_ops = len(mod_test.functions[0].ops)
n_params = len(mod_test.functions[0].params)

print(f"PES eigsolve module: {n_ops} ops, {n_params} params")
print(f"Requesting {n_states} eigenvalues (S0..S{n_states - 1})")
print(f"IR size: {len(ir_text)} chars")
print(f"\nFirst 400 chars of IR:\n{ir_text[:400]}...")

## 5. Preflight: Hardware Assignment

Query uniqx for Pareto-optimal execution options. At small qubit counts the
eigensolver runs on CPU; at larger active spaces (6+ qubits) QPU or GPU backends
become competitive.

In [ ]:
options = preflight(mod_test, client=client)
print(f"Pre-flight options (job_id={options.job_id}):\n")

for opt in options:
    rec = "  <-- recommended" if opt.get("recommended") else ""
    print(
        f"  [{opt['_idx']}] {opt['label']}: "
        f"time={opt['total_time']:.1f}, cost=${opt['total_cost_usd']:.6f}{rec}"
    )
    assignments = opt.get("node_assignments", {})
    if assignments:
        targets = set(assignments.values())
        print(f"      targets: {', '.join(sorted(targets))}")
    print()

SELECTED_OPTION = options.recommended["_idx"] if options.recommended else 0
print(f"Selected option: {SELECTED_OPTION}")

## 6. PES Scan: Compute Energies at Each Torsion Angle

Sweep the torsion angle from 0° to 180° in `ANGLE_STEP` increments. At each
geometry, build the Hamiltonian, trace the eigsolve module, submit to uniqx,
and collect the eigenvalues.

The result is a set of adiabatic energy curves: $E_{S0}(\theta), E_{S1}(\theta), \ldots$

In [ ]:
angles = list(range(0, 181, ANGLE_STEP))
n_points = len(angles)

# Storage: rows of eigenvalues per angle
energies = {f"S{i}": [] for i in range(n_states)}
diabatic_s0 = []
diabatic_s1 = []
couplings = []

print(f"PES scan: {n_points} points from 0 to 180 deg (step={ANGLE_STEP})")
print(f"Computing {n_states} adiabatic states per point...\n")

t0 = time.monotonic()

for idx, angle in enumerate(angles):
    H_theta, dim_h, e0_dia, e1_dia, V_theta = build_torsion_hamiltonian(
        float(angle), p, n_qubits
    )
    diabatic_s0.append(e0_dia)
    diabatic_s1.append(e1_dia)
    couplings.append(V_theta)

    mod = pes_eigsolve(H_theta)
    jid = submit(
        mod,
        client=client,
        runtime_inputs=[fmt_mat(H_theta, dim_h, dim_h)],
        preflight_job_id=options.job_id,
        option_idx=SELECTED_OPTION,
    )
    res = get(jid, client=client)

    payload = res.get("payload", b"")
    if isinstance(payload, str):
        payload = payload.encode()
    out = parse_result(payload, ["eigenvalues"])
    evals = out["eigenvalues"][2]
    levels = distinct_energy_levels(evals, n_states)

    for i in range(n_states):
        energies[f"S{i}"].append(levels[i])

    if (idx + 1) % 10 == 0 or idx == 0:
        evals_str = ", ".join(f"{e:.4f}" for e in evals[:n_states])
        print(f"  [{idx + 1}/{n_points}] theta={angle:>3d} deg: [{evals_str}] eV")

dt_scan = time.monotonic() - t0
print(f"\nPES scan complete in {dt_scan:.2f}s ({dt_scan / n_points:.3f}s per point)")

# Summary
s0 = energies["S0"]
s1 = energies["S1"]
gap_at_fc = s1[0] - s0[0]
ci_idx = min(range(len(angles)), key=lambda i: abs(s1[i] - s0[i]))
gap_at_ci = abs(s1[ci_idx] - s0[ci_idx])

print(f"\nVertical excitation (FC):  {gap_at_fc:.3f} eV ({1239.8 / gap_at_fc:.0f} nm)")
print(f"Gap at CI ({angles[ci_idx]} deg):       {gap_at_ci:.3f} eV")
print(f"S0 barrier (max):         {max(s0):.3f} eV")
print(f"S1 minimum:               {min(s1):.3f} eV")

## 7. Plot: Potential Energy Surfaces

The main result: adiabatic PES showing the photochemical reaction pathway.

The plot annotates:
- **FC** (Franck-Condon region): where vertical excitation occurs at equilibrium
- **CI** (conical intersection): where S0 and S1 approach, enabling non-adiabatic decay
- **Reaction pathway**: absorption → relaxation on S1 → decay through CI → product on S0

In [ ]:
STATE_COLORS = ["#2563eb", "#f59e0b", "#ef4444", "#8b5cf6"]
STATE_LABELS = ["S\u2080", "S\u2081", "S\u2082", "S\u2083"]

fig, ax = plt.subplots(figsize=(12, 7))

# Franck-Condon region shading
ax.axvspan(0, 15, color="#fef3c7", alpha=0.6, label="Franck-Condon")

# Conical intersection region shading
ci_lo = mol["ci_angle"] - 8
ci_hi = mol["ci_angle"] + 8
ax.axvspan(ci_lo, ci_hi, color="#fecaca", alpha=0.5, label="Conical intersection")

# CI dashed line
ax.axvline(mol["ci_angle"], color="#dc2626", linestyle="--", linewidth=0.8, alpha=0.6)

# Plot adiabatic states
for i in range(n_states):
    key = f"S{i}"
    ax.plot(
        angles,
        energies[key],
        color=STATE_COLORS[i],
        linewidth=2.5,
        label=STATE_LABELS[i],
        zorder=3,
    )

# Plot diabatic curves (dashed, thin)
ax.plot(
    angles,
    diabatic_s0,
    color="#93c5fd",
    linestyle=":",
    linewidth=1.0,
    alpha=0.5,
    label="S\u2080 diabatic",
)
ax.plot(
    angles,
    diabatic_s1,
    color="#fcd34d",
    linestyle=":",
    linewidth=1.0,
    alpha=0.5,
    label="S\u2081 diabatic",
)

# Vertical excitation arrow at FC
ax.annotate(
    "",
    xy=(3, s1[0]),
    xytext=(3, s0[0]),
    arrowprops=dict(arrowstyle="->", color="#d97706", lw=2.0),
)
ax.text(
    6,
    (s0[0] + s1[0]) / 2,
    f"h\u03bd\n{gap_at_fc:.2f} eV\n({1239.8 / gap_at_fc:.0f} nm)",
    fontsize=9,
    color="#92400e",
    ha="left",
    va="center",
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#fef3c7", edgecolor="#fbbf24", alpha=0.8
    ),
)

# CI gap annotation
ax.annotate(
    f"\u0394E = {gap_at_ci:.3f} eV",
    xy=(angles[ci_idx], (s0[ci_idx] + s1[ci_idx]) / 2),
    xytext=(angles[ci_idx] + 15, (s0[ci_idx] + s1[ci_idx]) / 2 + 0.5),
    fontsize=9,
    color="#991b1b",
    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=1.5),
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#fecaca", edgecolor="#f87171", alpha=0.8
    ),
)

# Product labels
ax.text(2, s0[0] - 0.3, "cis", fontsize=10, color="#1e40af", fontweight="bold")
ax.text(172, s0[-1] - 0.3, "trans", fontsize=10, color="#1e40af", fontweight="bold")

# Region labels
ax.text(
    7,
    ax.get_ylim()[1] * 0.97,
    "FC",
    fontsize=11,
    color="#92400e",
    fontweight="bold",
    ha="center",
    va="top",
    transform=ax.get_yaxis_transform(),
)
ax.text(
    mol["ci_angle"],
    ax.get_ylim()[1] * 0.97,
    "CI",
    fontsize=11,
    color="#991b1b",
    fontweight="bold",
    ha="center",
    va="top",
    transform=ax.get_yaxis_transform(),
)

ax.set_xlabel("Torsion angle (\u00b0)", fontsize=13)
ax.set_ylabel("Energy (eV)", fontsize=13)
ax.set_title(
    f"{mol['label']} Photoisomerization PES\n"
    f"{BASIS.upper()} / {n_qubits}-qubit active space",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlim(0, 180)
ax.set_xticks(range(0, 181, 30))
ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 8. Energy Gap & Coupling Analysis

Plot the S1–S0 energy gap and the non-adiabatic coupling strength along the
torsion coordinate. The minimum gap identifies the conical intersection region
where non-radiative decay is most efficient.

In [ ]:
gaps = [abs(s1[i] - s0[i]) for i, _ in enumerate(angles)]

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)

# Top: energy gap
ax1.plot(angles, gaps, "g-", linewidth=2.0, label="S\u2081\u2013S\u2080 gap")
ax1.axhline(
    gap_at_ci,
    color="gray",
    linestyle=":",
    alpha=0.5,
    label=f"Min gap = {gap_at_ci:.3f} eV",
)
ax1.axvspan(ci_lo, ci_hi, color="#fecaca", alpha=0.3)
ax1.fill_between(angles, 0, gaps, color="#bbf7d0", alpha=0.3)
ax1.set_ylabel("S\u2081\u2013S\u2080 gap (eV)", fontsize=12)
ax1.set_title(f"Energy Gap & Non-Adiabatic Coupling \u2014 {mol['label']}", fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.2)

# Convert gap to wavelength for secondary axis
ax1_r = ax1.twinx()
wavelengths = [1239.8 / g if g > 0.01 else np.nan for g in gaps]
ax1_r.plot(angles, wavelengths, "m--", linewidth=1.0, alpha=0.5)
ax1_r.set_ylabel("Wavelength (nm)", fontsize=10, color="purple")
ax1_r.tick_params(axis="y", labelcolor="purple")

# Bottom: coupling
ax2.plot(angles, couplings, "r-", linewidth=2.0, label="Coupling V(\u03b8)")
ax2.fill_between(angles, 0, couplings, color="#fecaca", alpha=0.3)
ax2.set_xlabel("Torsion angle (\u00b0)", fontsize=12)
ax2.set_ylabel("Coupling (eV)", fontsize=12)
ax2.set_xlim(0, 180)
ax2.set_xticks(range(0, 181, 30))
ax2.legend(fontsize=10)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print(f"Minimum gap: {min(gaps):.4f} eV at {angles[gaps.index(min(gaps))]} deg")
print(
    f"Maximum coupling: {max(couplings):.4f} eV at {angles[couplings.index(max(couplings))]} deg"
)

# Absorption wavelength
lambda_abs = 1239.8 / gap_at_fc
print(f"\nAbsorption wavelength: {lambda_abs:.0f} nm", end="")
if lambda_abs < 400:
    print(" (UV)")
elif lambda_abs < 700:
    print(" (visible)")
else:
    print(" (IR)")

## 9. Basis Set Comparison

Compute the PES with all three basis sets and overlay them to show how
basis set quality affects the predicted excitation energy and CI gap.

In [ ]:
basis_sets = ["sto-3g", "6-31g", "cc-pvdz"]
basis_colors = {"sto-3g": "#6366f1", "6-31g": "#10b981", "cc-pvdz": "#f43f5e"}
basis_results = {}

# Use coarser step for comparison (faster)
angles_cmp = list(range(0, 181, 10))

print("Basis set comparison scan...\n")

for bs in basis_sets:
    p_bs = mol["params"][bs]
    s0_bs, s1_bs = [], []

    for angle in angles_cmp:
        H_bs, dim_bs, _, _, _ = build_torsion_hamiltonian(float(angle), p_bs, n_qubits)
        mod_bs = pes_eigsolve(H_bs)
        jid = submit(
            mod_bs,
            client=client,
            runtime_inputs=[fmt_mat(H_bs, dim_bs, dim_bs)],
        )
        res = get(jid, client=client)
        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["eigenvalues"])
        evals = out["eigenvalues"][2]
        levels_bs = distinct_energy_levels(evals, 2)
        s0_bs.append(levels_bs[0])
        s1_bs.append(levels_bs[1])

    basis_results[bs] = {"S0": s0_bs, "S1": s1_bs}
    gap_fc = s1_bs[0] - s0_bs[0]
    ci_idx_bs = min(range(len(angles_cmp)), key=lambda i: abs(s1_bs[i] - s0_bs[i]))
    gap_ci_bs = abs(s1_bs[ci_idx_bs] - s0_bs[ci_idx_bs])
    print(
        f"  {bs:>8}: vert. exc. = {gap_fc:.3f} eV ({1239.8 / gap_fc:.0f} nm), "
        f"CI gap = {gap_ci_bs:.3f} eV at {angles_cmp[ci_idx_bs]} deg"
    )

# Plot comparison
fig, ax = plt.subplots(figsize=(11, 6))

for bs in basis_sets:
    c = basis_colors[bs]
    ax.plot(
        angles_cmp,
        basis_results[bs]["S0"],
        color=c,
        linewidth=2.0,
        label=f"S\u2080 ({bs})",
    )
    ax.plot(
        angles_cmp,
        basis_results[bs]["S1"],
        color=c,
        linewidth=2.0,
        linestyle="--",
        label=f"S\u2081 ({bs})",
    )

ax.axvspan(ci_lo, ci_hi, color="#fecaca", alpha=0.3, label="CI region")

ax.set_xlabel("Torsion angle (\u00b0)", fontsize=13)
ax.set_ylabel("Energy (eV)", fontsize=13)
ax.set_title(
    f"Basis Set Comparison: {mol['label']}\n"
    f"S\u2080 (solid) and S\u2081 (dashed) for STO-3G / 6-31G / cc-pVDZ",
    fontsize=13,
    fontweight="bold",
)
ax.set_xlim(0, 180)
ax.set_xticks(range(0, 181, 30))
ax.legend(fontsize=9, ncol=2, loc="upper right")
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 10. Photochemical Pathway Diagram

An annotated diagram tracing the four stages of the photoisomerization:

1. **Absorption** — vertical (Franck-Condon) excitation from S0 to S1 at 0°
2. **Relaxation** — the molecule twists on the S1 surface toward the CI (∼90°)
3. **Non-adiabatic decay** — ultrafast transition through the CI back to S0
4. **Product formation** — relaxation on S0 yields cis (0°) or trans (180°) isomer

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

# Background shading
ax.axvspan(0, 15, color="#fef3c7", alpha=0.5)
ax.axvspan(ci_lo, ci_hi, color="#fecaca", alpha=0.4)

# PES curves
ax.plot(angles, s0, color="#2563eb", linewidth=3.0, label="S\u2080", zorder=3)
ax.plot(angles, s1, color="#f59e0b", linewidth=3.0, label="S\u2081", zorder=3)

# Stage 1: Absorption (vertical arrow at FC)
ax.annotate(
    "",
    xy=(3, s1[0]),
    xytext=(3, s0[0]),
    arrowprops=dict(arrowstyle="->", color="#d97706", lw=2.5, mutation_scale=15),
    zorder=4,
)
ax.text(
    8,
    (s0[0] + s1[0]) / 2,
    "\u2460 Absorption\n" + f"h\u03bd = {gap_at_fc:.2f} eV",
    fontsize=10,
    color="#92400e",
    fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#fef3c7", edgecolor="#fbbf24"),
)

# Stage 2: Relaxation on S1 (curved arrow along S1)
mid_relax = len(angles) // 4  # roughly 45 deg
ax.annotate(
    "\u2461 Relaxation on S\u2081",
    xy=(angles[ci_idx] - 10, s1[ci_idx] + 0.1),
    xytext=(40, s1[mid_relax] + 0.6),
    fontsize=10,
    color="#b45309",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="->", color="#f59e0b", lw=2.0, connectionstyle="arc3,rad=-0.2"
    ),
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#fef9c3", edgecolor="#fbbf24", alpha=0.9
    ),
)

# Stage 3: Decay through CI (vertical arrow at CI)
ax.annotate(
    "",
    xy=(angles[ci_idx], s0[ci_idx]),
    xytext=(angles[ci_idx], s1[ci_idx]),
    arrowprops=dict(arrowstyle="->", color="#dc2626", lw=2.5, mutation_scale=15),
    zorder=4,
)
ax.text(
    angles[ci_idx] + 5,
    (s0[ci_idx] + s1[ci_idx]) / 2 + 0.3,
    "\u2462 Non-adiabatic\n    decay",
    fontsize=10,
    color="#991b1b",
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#fecaca", edgecolor="#f87171", alpha=0.9
    ),
)

# Stage 4: Product formation (arrows to cis and trans on S0)
ax.annotate(
    "\u2463 trans",
    xy=(170, s0[-1]),
    xytext=(angles[ci_idx] + 30, s0[ci_idx] - 0.5),
    fontsize=10,
    color="#1e40af",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="->", color="#2563eb", lw=2.0, connectionstyle="arc3,rad=-0.15"
    ),
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#dbeafe", edgecolor="#60a5fa", alpha=0.9
    ),
)
ax.annotate(
    "\u2463 cis",
    xy=(10, s0[0]),
    xytext=(angles[ci_idx] - 30, s0[ci_idx] - 0.5),
    fontsize=10,
    color="#1e40af",
    fontweight="bold",
    arrowprops=dict(
        arrowstyle="->", color="#2563eb", lw=2.0, connectionstyle="arc3,rad=0.15"
    ),
    bbox=dict(
        boxstyle="round,pad=0.3", facecolor="#dbeafe", edgecolor="#60a5fa", alpha=0.9
    ),
)

# CI marker
ax.plot(
    angles[ci_idx],
    (s0[ci_idx] + s1[ci_idx]) / 2,
    "*",
    color="#dc2626",
    markersize=18,
    zorder=5,
)

ax.set_xlabel("Torsion angle (\u00b0)", fontsize=13)
ax.set_ylabel("Energy (eV)", fontsize=13)
ax.set_title(
    f"Photoisomerization Pathway: {mol['label']}\n"
    f"{BASIS.upper()} / {n_qubits}-qubit active space",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlim(0, 180)
ax.set_xticks(range(0, 181, 30))
ax.legend(loc="upper left", fontsize=11)
ax.grid(alpha=0.15)

plt.tight_layout()
plt.show()

## 11. Results Summary

In [ ]:
print(f"Photoisomerization PES Results: {mol['label']}")
print("=" * 70)
print()

print("Configuration:")
print(f"  Molecule:       {mol['label']}")
print(f"  Basis set:      {BASIS.upper()}")
print(f"  Active space:   {n_qubits} qubits (dim={dim})")
print(f"  States:         {n_states} (S0..S{n_states - 1})")
print(f"  Angle points:   {n_points} (step={ANGLE_STEP} deg)")
print()

print("Photophysics:")
print(
    f"  Vertical excitation (S0\u2192S1): {gap_at_fc:.3f} eV ({1239.8 / gap_at_fc:.0f} nm)"
)
lambda_abs = 1239.8 / gap_at_fc
region = "UV" if lambda_abs < 400 else ("visible" if lambda_abs < 700 else "IR")
print(f"  Absorption region:           {region}")
print(f"  S0 barrier at 90\u00b0:           {max(s0):.3f} eV")
print(f"  S1 minimum (adiabatic):      {min(s1):.3f} eV")
print(f"  CI gap:                      {gap_at_ci:.4f} eV at {angles[ci_idx]}\u00b0")
print(f"  S0 trans offset:             {s0[-1] - s0[0]:.3f} eV")
print()

print("Computation:")
print(f"  Total time:     {dt_scan:.2f}s")
print(f"  Per point:      {dt_scan / n_points:.3f}s")
print(f"  uniqx jobs:   {n_points}")
hw = "QPU" if n_qubits >= 6 else "CPU"
print(f"  Primary target: {hw} (via uniqx routing)")
print()

print("Photochemical pathway:")
print(f"  1. Absorption:    S0 \u2192 S1 at 0\u00b0 (FC), \u0394E = {gap_at_fc:.2f} eV")
print(f"  2. Relaxation:    S1 surface, 0\u00b0 \u2192 ~{angles[ci_idx]}\u00b0")
print(f"  3. Decay:         S1 \u2192 S0 through CI (\u0394E = {gap_at_ci:.3f} eV)")
print("  4. Product:       cis (0\u00b0) or trans (180\u00b0) on S0")

## Summary

| Aspect | Detail |
|:-------|:-------|
| **Problem** | Cis-trans photoisomerization via torsional PES scan |
| **Molecules** | Ethylene, formaldimine, butadiene, acrolein |
| **Method** | Active-space eigensolver at each torsion geometry |
| **Primitives** | `eigs` (eigendecomposition → CPU/GPU/QPU) |
| **Visualisation** | Adiabatic PES, energy gap, coupling profile, pathway diagram |
| **Key physics** | Franck-Condon excitation, conical intersection, non-adiabatic decay |

**Key takeaways:**

1. **Conical intersections drive photochemistry**: The PES scan reveals the conical intersection where S0 and S1 nearly touch, enabling ultrafast non-radiative decay. This geometry determines the quantum yield of photoisomerization.

2. **Basis set matters**: Larger basis sets lower the excitation energy and tighten the CI gap. The cc-pVDZ basis gives qualitatively correct results for small molecules.

3. **Biological relevance**: Formaldimine is the minimal model of the retinal protonated Schiff base — the chromophore in rhodopsin. The same photoisomerization mechanism powers biological vision.

4. **Hardware routing**: uniqx routes `eigs` to the best available hardware. At 4 qubits, CPU/GPU dominates; at 6+ qubits, QPE on QPU becomes competitive.

## 12. Comparison with PySCF (CIS Excited States)

PySCF provides excited-state methods including CIS (Configuration Interaction Singles)
and TDDFT. We compare the uniqx model Hamiltonian results against PySCF CIS for the
same molecule to validate:

1. **Vertical excitation energies** (S0 -> S1 gap at the Franck-Condon point)
2. **Ground-state energy** at the equilibrium (planar) geometry
3. **Timing** for a single-point excited-state calculation

Since the uniqx notebook uses a parameterised model Hamiltonian rather than
a full ab-initio calculation, exact numerical agreement is not expected.
Instead, we verify that the qualitative features (excitation energy scale,
ordering of states) are physically reasonable compared to first-principles CIS.

In [ ]:
%%time

# --- PySCF CIS comparison for photoisomerization ---
try:
    from pyscf import gto, scf, tdscf
    PYSCF_AVAILABLE = True
except ImportError:
    PYSCF_AVAILABLE = False
    print("PySCF not installed. Install with: pip install pyscf")
    print("Skipping PySCF comparison.")

pyscf_excited = {}

if PYSCF_AVAILABLE:
    import time as _time

    # Molecule geometries for PySCF (planar equilibrium)
    PYSCF_MOLECULES = {
        "ethylene": "C 0 0 0; C 1.339 0 0; H -0.5 0.93 0; H -0.5 -0.93 0; H 1.839 0.93 0; H 1.839 -0.93 0",
        "formaldimine": "C 0 0 0; N 1.25 0 0; H -0.55 0.93 0; H -0.55 -0.93 0; H 1.8 0.88 0",
        "butadiene": "C 0 0 0; C 1.34 0 0; C 2.46 0.7 0; C 3.80 0.7 0; H -0.55 0.93 0; H -0.55 -0.93 0; H 1.89 -0.93 0; H 2.01 1.63 0; H 4.35 1.63 0; H 4.35 -0.23 0",
        "acrolein": "C 0 0 0; C 1.34 0 0; C 2.46 0.7 0; O 3.65 0.7 0; H -0.55 0.93 0; H -0.55 -0.93 0; H 1.89 -0.93 0; H 2.01 1.63 0",
    }

    mol_geom = PYSCF_MOLECULES.get(MOLECULE)
    if mol_geom is None:
        print(f"No PySCF geometry defined for {MOLECULE}, skipping.")
    else:
        basis_map = {"sto-3g": "sto-3g", "6-31g": "6-31g", "cc-pvdz": "cc-pvdz"}
        pyscf_basis = basis_map.get(BASIS, "sto-3g")

        mol_pyscf = gto.M(
            atom=mol_geom,
            basis=pyscf_basis,
            unit="Angstrom",
            verbose=0,
        )

        # --- Ground-state SCF ---
        t0 = _time.monotonic()
        mf = scf.RHF(mol_pyscf)
        mf.kernel()
        t_scf_pyscf = _time.monotonic() - t0

        print(f"PySCF RHF energy ({MOLECULE}, {pyscf_basis}): {mf.e_tot:.6f} Ha")

        # --- CIS / TDHF excited states ---
        t0 = _time.monotonic()
        td = tdscf.TDA(mf)  # CIS = TDA (Tamm-Dancoff Approximation)
        td.nstates = N_EXCITED
        td.verbose = 0
        td.kernel()
        t_cis_pyscf = _time.monotonic() - t0

        # Extract excitation energies in eV
        cis_excitations_ev = [e * 27.2114 for e in td.e]  # Ha to eV

        pyscf_excited["scf_energy"] = mf.e_tot
        pyscf_excited["cis_excitations_ev"] = cis_excitations_ev
        pyscf_excited["t_scf"] = t_scf_pyscf
        pyscf_excited["t_cis"] = t_cis_pyscf

        print(f"\nPySCF CIS excitation energies ({t_cis_pyscf:.3f}s):")
        for i, e_ev in enumerate(cis_excitations_ev):
            print(f"  S0 -> S{i + 1}: {e_ev:.3f} eV ({1240.0 / e_ev:.0f} nm)")

        # Compare with uniqx model at 0 degrees (Franck-Condon)
        s0_fc = energies["S0"][0]
        s1_fc = energies["S1"][0]
        uniqx_excitation = s1_fc - s0_fc

        print(f"\nVertical excitation energy comparison (S0 -> S1 at FC point):")
        print(f"  uniqx model:  {uniqx_excitation:.3f} eV")
        print(f"  PySCF CIS:    {cis_excitations_ev[0]:.3f} eV")
        print(f"  Difference:   {abs(uniqx_excitation - cis_excitations_ev[0]):.3f} eV")

In [ ]:
if PYSCF_AVAILABLE and pyscf_excited:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # --- Left: Excitation energy comparison (bar chart) ---
    n_exc = min(len(cis_excitations_ev), N_EXCITED)
    state_labels = [f"S0->S{i + 1}" for i in range(n_exc)]
    uniqx_exc = []
    for i in range(n_exc):
        s_key = f"S{i + 1}"
        if s_key in energies and len(energies[s_key]) > 0:
            uniqx_exc.append(energies[s_key][0] - energies["S0"][0])
        else:
            uniqx_exc.append(0.0)

    x = np.arange(len(state_labels))
    width = 0.35
    axes[0].bar(x - width / 2, uniqx_exc, width, label="uniqx (model H)", color="#2563eb", alpha=0.8)
    axes[0].bar(x + width / 2, cis_excitations_ev[:n_exc], width, label="PySCF CIS", color="#16a34a", alpha=0.8)
    axes[0].set_xlabel("Transition")
    axes[0].set_ylabel("Excitation energy (eV)")
    axes[0].set_title(f"{MOLECULE} Vertical Excitations ({BASIS})")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(state_labels)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    # --- Middle: PES from uniqx with PySCF CIS excitation marked ---
    axes[1].plot(angles, energies["S0"], "b-", linewidth=2.0, label="S0 (uniqx)")
    axes[1].plot(angles, energies["S1"], "r-", linewidth=2.0, label="S1 (uniqx)")
    if N_EXCITED >= 2 and "S2" in energies and len(energies["S2"]) > 0:
        axes[1].plot(angles, energies["S2"], "g-", linewidth=1.5, label="S2 (uniqx)", alpha=0.7)

    # Mark PySCF CIS excitations at 0 degrees
    for i, e_cis in enumerate(cis_excitations_ev[:n_exc]):
        e_abs = energies["S0"][0] + e_cis
        axes[1].axhline(y=e_abs, color="green", linestyle="--", alpha=0.4)
        axes[1].annotate(
            f"PySCF S{i + 1} ({e_cis:.2f} eV)",
            xy=(5, e_abs),
            fontsize=8,
            color="green",
        )

    axes[1].set_xlabel("Torsion angle (deg)")
    axes[1].set_ylabel("Energy (eV)")
    axes[1].set_title(f"{MOLECULE} PES with PySCF Reference")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    # --- Right: Timing ---
    t_uniqx_pes = dt_scan  # from the PES scan above
    t_pyscf = pyscf_excited["t_scf"] + pyscf_excited["t_cis"]

    bar_labels_t = [f"uniqx PES\n({n_points} pts)", "PySCF\n(1 pt, CIS)"]
    bar_vals_t = [t_uniqx_pes, t_pyscf]
    bar_colors_t = ["#2563eb", "#16a34a"]
    axes[2].bar(bar_labels_t, bar_vals_t, color=bar_colors_t, alpha=0.8, edgecolor="black")
    axes[2].set_ylabel("Wall time (s)")
    axes[2].set_title("Timing Comparison")
    axes[2].grid(axis="y", alpha=0.3)
    for i, v in enumerate(bar_vals_t):
        axes[2].text(i, v + max(bar_vals_t) * 0.02, f"{v:.3f}s", ha="center", fontsize=10)

    fig.suptitle(
        f"{MOLECULE} Photoisomerization: uniqx vs PySCF CIS ({BASIS})",
        fontsize=14, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

    # --- Literature reference values ---
    LITERATURE = {
        "ethylene": {"S0_S1_eV": 7.66, "source": "Merer & Mulliken, Chem. Rev. 69, 639 (1969)"},
        "formaldimine": {"S0_S1_eV": 5.7, "source": "Bonacic-Koutecky et al., Theor. Chim. Acta 68, 45 (1985)"},
        "butadiene": {"S0_S1_eV": 5.92, "source": "Doering et al., JACS 102, 3780 (1980)"},
        "acrolein": {"S0_S1_eV": 3.69, "source": "Walsh, JACS 63, 2179 (1946) [n->pi*]"},
    }

    if MOLECULE in LITERATURE:
        ref = LITERATURE[MOLECULE]
        print(f"\nLiterature reference for {MOLECULE}:")
        print(f"  S0->S1 excitation: {ref['S0_S1_eV']:.2f} eV")
        print(f"  Source: {ref['source']}")
        print(f"  uniqx model:  {uniqx_excitation:.3f} eV")
        print(f"  PySCF CIS:    {cis_excitations_ev[0]:.3f} eV")
else:
    print("Skipping comparison plots (PySCF not available or no results).")